# Volume 9, Chapter 10: Shuffling Cards (Python companion)

The exact law of random transpositions from the characters of the symmetric group, the cutoff, the upper bound lemma for a 52-card deck, moments of fixed points, and how (not) to shuffle in a program.

*Before each solution, predict what the output will look like.*

## Exercises

**P10.1 (★★) The exact distance to uniform.** Implement the characters of $\mathfrak S_n$ (Murnaghan–Nakayama rule, which removes rim hooks; we use it here as a black box) and the formula of Proposition 10.5 for the law of the deck after $t$ random transpositions. (a) Check it against a brute-force computation of the Markov chain on $\mathfrak S_5$. (b) Compute the exact $d(t)$ for $n=10,15,20$ and plot it against $c=(t-\frac12n\ln n)/n$ (Figure 10.1).

**P10.2 (★) The upper bound lemma for a real deck.** For $n=52$, compute the bound $\frac12\sqrt S$ of Proposition 10.5 by summing over all $281\,589$ partitions of $52$, and find the first $t$ at which it drops below $\frac14$ and below $0.01$. Which shapes contribute most?

**P10.3 (★) Moments of fixed points.** Check by simulation the formulas of Exercises 10.4 and 10.6 for $\mathbb E[F]$ and $\mathbb E[F(F-1)]$ after $t$ swaps of $52$ cards, and compute the exact variance of $F(X_t)$.

**P10.4 (★★) CS application: shuffling in a program.** A program that needs a random permutation (a card game, a randomised algorithm, a statistical test) should not perform random transpositions: $\frac12n\ln n$ swaps are needed, and even then the result is only approximately uniform. The Fisher–Yates algorithm (Durstenfeld, 1964) swaps position $i$ with a uniform position $j\le i$, for $i=n-1,\dots,1$, and is *exactly* uniform with $n-1$ swaps. (a) Prove it, and check it for $n=4$. (b) A classic bug swaps position $i$ with a uniform position $j$ among *all* $n$ positions. Compute the exact law it produces for $n=3$ and $n=4$, and its distance to uniform for $n\le8$.

## Solutions

### P10.1

In [ ]:
import math, itertools, functools
from collections import Counter
import numpy as np

def partitions(n, maxpart=None):
    maxpart = n if maxpart is None else maxpart
    if n == 0: yield (); return
    for k in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - k, k): yield (k,) + rest

def conjugate(lam): return tuple(sum(1 for x in lam if x > j) for j in range(lam[0])) if lam else ()

def hook_formula(lam):
    c = conjugate(lam); p = 1
    for i, x in enumerate(lam):
        for j in range(x): p *= (x - j - 1) + (c[j] - i - 1) + 1
    return math.factorial(sum(lam)) // p

@functools.lru_cache(None)
def mn(beta, mu):
    """Murnaghan-Nakayama on beta-sets: remove a rim hook of length mu[0] in all possible ways."""
    if not mu: return 1
    r, total = mu[0], 0
    for b in beta:
        if b - r >= 0 and b - r not in beta:
            sign = (-1) ** sum(1 for x in beta if b - r < x < b)
            total += sign * mn((beta - {b}) | {b - r}, mu[1:])
    return total

def character(lam, mu):
    L = len(lam)
    return mn(frozenset(lam[i] + L - 1 - i for i in range(L)), tuple(mu))

def class_size(mu):
    z = 1
    for k, m in Counter(mu).items(): z *= k ** m * math.factorial(m)
    return math.factorial(sum(mu)) // z

def s_eig(lam):
    n = sum(lam)
    contents = sum(j - i for i, x in enumerate(lam) for j in range(x))
    return 1 / n + (n - 1) / n * contents / math.comb(n, 2)

def exact_distance(n, ts):
    P = list(partitions(n))
    chi = {(l, m): character(l, m) for l in P for m in P}
    f = {l: hook_formula(l) for l in P}; s = {l: s_eig(l) for l in P}
    out = []
    for t in ts:
        d = 0.0
        for mu in P:
            p = sum(f[l] * chi[l, mu] * s[l] ** t for l in P) / math.factorial(n)
            d += class_size(mu) * abs(p - 1 / math.factorial(n))
        out.append(d / 2)
    return out

# (a) brute force on S_5: two independent uniform cards are swapped
n = 5
G = list(itertools.permutations(range(n))); idx = {g: i for i, g in enumerate(G)}
T = np.zeros((len(G), len(G)))
for g in G:
    for a in range(n):
        for b in range(n):
            h = list(g); h[a], h[b] = h[b], h[a]
            T[idx[g], idx[tuple(h)]] += 1 / n ** 2
p = np.zeros(len(G)); p[idx[tuple(range(n))]] = 1; brute = []
for t in range(10):
    brute.append(0.5 * np.abs(p - 1 / len(G)).sum()); p = p @ T
print("brute force:", np.round(brute, 6))
print("characters: ", np.round(exact_distance(5, range(10)), 6))

The two computations agree to six decimals: the Fourier formula is exact. Part (b): the characters of $\mathfrak S_{20}$ form a $627\times627$ table, which the cached rule computes in a few seconds.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
for n in (10, 15, 20):
    ts = list(range(0, int(0.5 * n * math.log(n) + 3 * n)))
    d = exact_distance(n, ts)
    plt.plot([(t - 0.5 * n * math.log(n)) / n for t in ts], d, label=f"n = {n}")
    t_mix = next(t for t, x in zip(ts, d) if x <= 0.25)
    print(f"n = {n}: t_mix = {t_mix},  (1/2) n ln n = {0.5 * n * math.log(n):.1f},  (t_mix - (1/2) n ln n)/n = {(t_mix - 0.5*n*math.log(n))/n:.2f}")
plt.xlabel("c = (t - n ln n / 2) / n"); plt.ylabel("d(t)"); plt.legend(); plt.grid(alpha=.3); plt.show()

In the variable $c$ the three curves are close and get closer as $n$ grows, and $t_{\mathrm{mix}}$ sits a fraction of $n$ above $\frac12n\ln n$, as Corollary 10.9 predicts: the cutoff is visible already at $n=10$.

### P10.2

In [ ]:
import time
n = 52
t0 = time.time()
parts = [l for l in partitions(n) if l != (n,)]
f = np.array([float(hook_formula(l)) for l in parts])
s = np.array([s_eig(l) for l in parts])
print(len(parts) + 1, "partitions, computed in", round(time.time() - t0, 1), "s")

def bound(t): return 0.5 * math.sqrt(np.sum(f ** 2 * np.abs(s) ** (2 * t)))
half = 0.5 * n * math.log(n)
for target in (0.25, 0.01):
    t = next(t for t in range(1, 2000) if bound(t) <= target)
    print(f"bound <= {target}: t = {t} = (1/2) n ln n + {(t - half) / n:.2f} n")

t = 155
terms = f ** 2 * np.abs(s) ** (2 * t)
order = np.argsort(-terms)
print(f"\nat t = {t}: S = {terms.sum():.4g}; largest terms:")
for i in order[:5]:
    print(f"  {parts[i]}: f = {int(f[i])}, s = {s[i]:+.4f}, term = {terms[i]:.3g}")

The bound guarantees $d\le\frac14$ after $120$ swaps and $d\le0.01$ after $200$, not far above $\frac12n\ln n\approx103$: for a real deck the representation theory gives a practical answer, not just an asymptotic one (the true $t_{\mathrm{mix}}$ is a little smaller, since the bound loses something in Cauchy–Schwarz). At $t=155$ almost all of $S$ comes from the single shape $(51,1)$, the standard representation, which records where each card is; then come $(50,2)$ and $(50,1,1)$, and the sign representation $(1^{52})$, whose eigenvalue is close to $-1$ (each real swap flips the sign of the permutation, and the shuffle stays put only with probability $\frac1{52}$). Of the $281\,589$ terms a handful matter, as the proof of Theorem 10.8 expects: the long first rows carry everything.

### P10.3

In [ ]:
rng = np.random.default_rng(10)
n, runs = 52, 20000
for t in (20, 60, 103, 155):
    F = np.empty(runs)
    for r in range(runs):
        deck = np.arange(n)
        L, R = rng.integers(0, n, t), rng.integers(0, n, t)
        for a, b in zip(L, R): deck[a], deck[b] = deck[b], deck[a]
        F[r] = np.sum(deck == np.arange(n))
    EF = 1 + (n - 1) * (1 - 2 / n) ** t
    EFF = (1 + 2 * (n - 1) * (1 - 2 / n) ** t + n * (n - 3) / 2 * (1 - 2 / n) ** (2 * t)
           + (n - 1) * (n - 2) / 2 * (1 - 4 / n) ** t)
    print(f"t = {t:3}:  E[F] sim {F.mean():6.3f} exact {EF:6.3f} |  E[F(F-1)] sim {np.mean(F*(F-1)):7.3f} exact {EFF:7.3f}"
          f" |  Var F exact {EFF + EF - EF**2:6.3f}")

The simulated moments match the formulas. As $t$ grows, the mean and the variance of $F$ both tend to $1$, the values for a uniform permutation (the number of fixed points of a uniform permutation is close to Poisson with mean $1$). At $t=103\approx\frac12n\ln n$ the mean is still about $1.9$: the first-order effect (cards left in place) is the last to disappear, which is the content of the lower bound.

### P10.4

In [ ]:
from fractions import Fraction

def law(n, shuffle_choices):
    """Exact law of the permutation produced by a sequence of swaps (i, list of possible j, uniform)."""
    dist = {tuple(range(n)): Fraction(1)}
    for i, choices in shuffle_choices:
        new = Counter()
        for perm, p in dist.items():
            for j in choices:
                q = list(perm); q[i], q[j] = q[j], q[i]
                new[tuple(q)] += p / len(choices)
        dist = new
    return dist

fisher_yates = lambda n: [(i, list(range(i + 1))) for i in range(n - 1, 0, -1)]
naive = lambda n: [(i, list(range(n))) for i in range(n)]

d = law(4, fisher_yates(4))
print("Fisher-Yates, n = 4:", len(d), "permutations, probabilities:", set(d.values()))
for n in (3, 4):
    d = law(n, naive(n))
    print(f"naive, n = {n}: {n**n} equally likely paths onto {math.factorial(n)} permutations; probabilities x {n**n}:",
          sorted(Counter(int(p * n ** n) for p in d.values()).items()))
for n in range(3, 9):
    d = law(n, naive(n))
    tv = sum(abs(p - Fraction(1, math.factorial(n))) for p in d.values()) / 2
    print(f"naive shuffle, n = {n}: distance to uniform {float(tv):.4f}")

**Proof of (a).** By induction on $n$: the first step puts in position $n-1$ a uniformly random card ($j$ uniform among $n$ positions), and the remaining steps are Fisher–Yates on the first $n-1$ positions, which by induction arrange the other cards uniformly, independently of the first choice. So each of the $n!$ outcomes has probability $\frac1n\cdot\frac1{(n-1)!}$. The computation confirms that all $24$ permutations of $4$ cards have probability $\frac1{24}$.

**The bug.** The naive version makes $n$ uniform choices among $n$, so it has $n^n$ equally likely runs. For $n=3$, $27$ runs cannot be split evenly among $6$ permutations, since $6\nmid27$: some permutations get $4$ runs and others $5$. In general $n!$ never divides $n^n$ for $n\ge3$ ($n-1$ divides $n!$ but is coprime to $n$), so the naive shuffle is *never* uniform, and the table shows that its distance to uniform even grows with $n$ in this range. Random transpositions are the same mistake taken further: uniform random swaps with no plan, which only approach uniformity after about $\frac12n\ln n$ of them.